In [1]:
from google.colab import files

files = files.upload()

Saving train.csv to train (3).csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
train = pd.read_csv("train.csv")

In [4]:
train.head(2)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


# Milestone 1

###Q1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

Ans: 814

In [5]:
f_dis = train['answer'].value_counts(ascending = False )
f_dis

,count
answer,
B,490
C,459
A,369
D,358
E,324


In [6]:
l = f_dis.to_list()

sum = l[0] + l[-1]
sum

814

###Q2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

Ans: 859

In [7]:
import string

string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [8]:
prompt = train['prompt'].str.lower()
prompt[0]

"pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options."

In [9]:
def nopunctuation(text):
  text = text.translate(str.maketrans('','',string.punctuation))
  return text

prompt = prompt.apply(nopunctuation)
prompt[0]

'pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options'

In [10]:
text = ' '.join(prompt)
text = text.split()
unique_words = set(text)

len(unique_words)

859

####Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

Ans: 3

In [11]:
prompt.head()

,prompt
0,pick the best possible answer what is martin h...
1,what is acceleratorbased lightion fusion
2,determine the correct option what is the term ...
3,select the most accurate option what is martin...
4,identify the correct statement what is the con...


In [12]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print(len(ENGLISH_STOP_WORDS))
print(ENGLISH_STOP_WORDS)

318
frozenset({'sometime', 'very', 'co', 'our', 'before', 'do', 'none', 'already', 'were', 'below', 'myself', 'is', 'nevertheless', 'towards', 'therefore', 'further', 'herein', 'much', 'thereupon', 'should', 'two', 'three', 'never', 'of', 'once', 'if', 'everywhere', 'about', 'per', 'even', 'get', 'amount', 'how', 'him', 'wherein', 'perhaps', 'then', 'afterwards', 'why', 'rather', 'there', 'amoungst', 'becoming', 'one', 'yourselves', 'where', 'than', 'thereby', 'down', 'anything', 'may', 'latter', 'either', 'put', 'each', 'serious', 'and', 'anywhere', 'has', 'but', 'have', 'her', 'for', 'nine', 'many', 'elsewhere', 'becomes', 'whoever', 'until', 'next', 'the', 'your', 'nobody', 'back', 'seeming', 'thereafter', 'under', 'against', 'five', 'formerly', 'whereafter', 're', 'cry', 'whereupon', 'while', 'system', 'whereby', 'along', 'however', 'my', 'would', 'nothing', 'had', 'give', 'amongst', 'sometimes', 'most', 'such', 'ever', 'became', 'an', 'whenever', 'someone', 'within', 'describe', '

In [13]:
def remove_stopwords(text):
  words = text.split()
  for word in words:
    if word in ENGLISH_STOP_WORDS:
      words.remove(word)
  char = ' '.join(words)
  return char

prompt = prompt.apply(remove_stopwords)
len(prompt[1].split())

4

###Q3. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

Ans: 2762

In [14]:
combined_text = pd.concat([train['prompt'], train['A'],train['B'],train['C'],train['D'],train['E']])  #combining prompts and options in a single series
combined_text[0]

,0
0,Pick the best possible answer: What is Martin ...
0,Martin Heidegger believes that humans exist wi...
0,Martin Heidegger believes that humans do not e...
0,Martin Heidegger does not believe in the exist...
0,Martin Heidegger believes that the relationshi...
0,Martin Heidegger believes that time is an illu...


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words = 'english')
tfidf.fit(combined_text)

TfidfVectorizer(stop_words='english')

In [16]:
train_tfidf = {}
cols = ['prompt' , 'A','B','C','D','E']

for col in cols:
  train_tfidf[col] = tfidf.transform(train[col])

train_tfidf.keys()

dict_keys(['prompt', 'A', 'B', 'C', 'D', 'E'])

In [17]:
len(tfidf.get_feature_names_out())

2762

###Q4. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

Ans: 0.2328

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

cos_prompt_A = cosine_similarity(train_tfidf['prompt'][0] , train_tfidf['A'][0])
cos_prompt_A[0][0].round(4)

np.float64(0.2328)

###Q5. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

Ans: 13.7

In [19]:
options = ['A', 'B', 'C', 'D', 'E']

cos = {}

for opt in options:
  cos[opt] = cosine_similarity(train_tfidf['prompt'] , train_tfidf[opt]).diagonal()

cos.keys()

dict_keys(['A', 'B', 'C', 'D', 'E'])

In [20]:
cos_df = pd.DataFrame(cos)
cos_df.head()

,A,B,C,D,E
0,0.232800,0.258421,0.498558,0.451256,0.169300
1,0.501327,0.501201,0.528616,0.473393,0.374950
2,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.247120,0.274316,0.529224,0.479013,0.179714
4,0.071975,0.073314,0.072467,0.209584,0.209584


In [21]:
idxmax = cos_df.idxmax(axis = 1)
idxmax.shape

(2000,)

In [22]:
count = 0

for x,y in zip(idxmax , train['answer']):
  if x == y:
    count+=1
round(count/2000*100 ,4)

13.7

###Q6. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

Ans: 1

In [23]:
def map_at3(prediction,correct):  #function to calculate MAP@3 score
  score = 0
  for pred, x in zip(prediction, correct):
    if x in pred:
      k = pred.index(x) + 1
      score += 1/k
  return round(score/len(correct) ,4)

In [24]:
pred = [['C','A','B']]
actual = ['C']

print(map_at3(pred,actual))

1.0


###Q7. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

Ans: 0.5

In [25]:
print(map_at3([['D','B','E']], ['B']))

0.5


###Q8. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

Ans : 0.4213

In [26]:
top3_ans = train['answer'].value_counts().index[0:3].tolist()
print(f"Top 3 most frequent answers: {top3_ans}\n")

top3_pred = [top3_ans] * len(train)
print(f"List of top3 preds: {top3_pred[0:5]}\n")

correct_ans = train['answer'].tolist()
print("MAP@3 for top 3 most frequent answers:",map_at3(top3_pred, correct_ans))

Top 3 most frequent answers: ['B', 'C', 'A']

List of top3 preds: [['B', 'C', 'A'], ['B', 'C', 'A'], ['B', 'C', 'A'], ['B', 'C', 'A'], ['B', 'C', 'A']]

MAP@3 for top 3 most frequent answers: 0.4213


###Q9. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

Ans:0.3112

In [27]:
predictions = []
for i in range(len(cos_df)):
  predictions.append(cos_df.iloc[i].sort_values(ascending = False).index[:3].tolist())

predictions[:5]

[['C', 'D', 'B'],
 ['C', 'A', 'B'],
 ['A', 'B', 'C'],
 ['C', 'D', 'B'],
 ['E', 'D', 'B']]

In [28]:
print(map_at3(predictions , train['answer']))

0.3112
